# uniCOIL (2022)
---
[[paper]](https://arxiv.org/pdf/2112.03669.pdf) <br>
uniCOIL = **UNI**fied **CO**ntent-specific **I**nverted **L**ist

Метод разработан исследователями из University of Waterloo (команда Jimmy Lin). Это модель для **Sparse Retrieval**, которая использует глубокое обучение для вычисления весов терминов в инвертированном индексе.

**Суть решения:** uniCOIL — это способ превратить стандартный Sparse Retrieval (как BM25) в обучаемую нейросетевую модель. Она заменяет классические формулы подсчета веса (TF-IDF) на веса, предсказанные BERT-ом, сохраняя при этом возможность использовать классические и эффективные Inverted Indexes.

### Контекст
В поиске долгое время доминировал BM25 (1990-е), работающий на пересечении ключевых слов. Затем появился Dense Retrieval (DPR, 2020), который сопоставляет скрытые векторы. Но у плотного поиска есть минусы: огромные требования к оперативной памяти для хранения векторов и сложность интерпретации. uniCOIL возвращает нас к разреженным векторам, но делает их «умными».

### Альтернативы
На момент появления uniCOIL существовали:
*   **BM25 (1990-е):** использует статистические веса (частота слова). Проблема: Lexical Gap (не понимает синонимы).
*   **DeepCT (2019):** использует BERT для предсказания весов терминов вместо TF, но не умеет в Query Expansion.
*   **DocT5Query (2019):** расширяет документ, генерируя возможные вопросы к нему с помощью T5. Но ранжирование все равно идет через BM25.
*   **COIL (2021):** предшественник метода. В нем для каждого токена хранился небольшой вектор (например, 32d). Это требовало кастомных индексов и было сложнее в реализации.
*   **SPLADE (2021):** строит разреженный вектор по всему словарю BERT (30к+ измерений). Очень мощно, но требует сложных механизмов регуляризации (FLOPS loss) для контроля разреженности.

### Идея
Авторы решили упростить COIL, доказав, что нам не нужны многомерные векторы для каждого токена в индексе. Достаточно одного скалярного веса, если мы правильно расширим документ. uniCOIL объединяет семантическое взвешивание терминов и расширение документа (челько через doc2query-T5) в единый пайплайн разреженного поиска.

### Архитектура
Модель строится на базе **Encoder-only** трансформера (обычно BERT-base).
1.  **Input:** Последовательность токенов текста.
2.  **Transformer Layers:** Извлечение контекстных эмбеддингов для каждого токена.
3.  **Weighting Head:** Линейный слой поверх последнего слоя BERT, который преобразует вектор токена в одно число (скаляр).
4.  **Activation:** ReLU, чтобы веса были строго положительными (необходимое условие для работы классических поисковых движков вроде Lucene).

### Алгоритм обучения
Используется архитектура **Siamese Network** и Contrastive Learning:
1.  Берется тройка: Query ($q$), Positive Document ($d^+$) и Hard Negative Document ($d^-$).
2.  Модель (один и тот же BERT) прогоняет через себя все три текста и выдает веса для каждого токена.
3.  Для каждой пары $(q, d)$ считается Relevance Score как сумма произведений весов совпадающих токенов: $score(q, d) = \sum_{t \in q \cap d} w_q(t) \cdot w_d(t)$.
4.  Минимизируется Negative Log-Likelihood (NLL) того, что позитивный документ окажется выше негативного в ранжировании.

### Алгоритм инференса
1.  **Document Expansion (Offline):** К документу добавляются новые токены с помощью DocT5Query. Это решает проблему Lexical Gap.
2.  **Indexing (Offline):** Расширенный документ прогоняется через uniCOIL. Каждый токен получает вес. Эти пары (token, weight) записываются в стандартный Inverted Index (например, через Pyserini).
3.  **Query Processing (Online):** Запрос прогоняется через ту же модель uniCOIL для получения весов токенов запроса.
4.  **Search:** Выполняется взвешенный поиск в инвертированном индексе. Итоговый скор — это просто Dot Product разреженных векторов.

### Результаты
*   **Эффективность:** uniCOIL занимает в 4-5 раз меньше места на диске, чем Dense Retrieval индексы (DPR), так как хранит только скаляры в инвертированном списке, а не float-векторы.
*   **Качество:** На датасете MS MARCO (стандарт для IR) модель показала MRR@10 около 0.351. Для сравнения: BM25 на том же датасете выдает ~0.184, а классический DPR ~0.31-0.34.
*   **Новизна:** Доказано, что переход от векторных представлений токенов (как в COIL) к скалярным весам не снижает точность поиска, если использовать предварительное расширение документов, что радикально упрощает архитектуру поисковых систем.

## 📝 Критический анализ

```markdown
# uniCOIL (2022)
---
[[paper]](https://arxiv.org/pdf/2112.03669.pdf) <br>
uniCOIL = **UNI**fied **CO**ntent-specific **I**nverted **L**ist

Метод разработан исследователями из University of Waterloo (команда Jimmy Lin). Это модель для Sparse Retrieval, использующая глубокое обучение для вычисления весов терминов в инвертированном индексе.

**Суть решения:** uniCOIL превращает стандартный Sparse Retrieval (как BM25) в обучаемую нейросетевую модель, заменяя классические формулы подсчета веса (TF-IDF) на веса, предсказанные BERT-ом, сохраняя возможность использования Inverted Indexes.

### Контекст
В поиске долгое время доминировал BM25 (1990-е), работающий на пересечении ключевых слов. Затем появился Dense Retrieval (DPR, 2020), который сопоставляет скрытые векторы. Однако у плотного поиска есть минусы: высокие требования к памяти и сложность интерпретации. uniCOIL возвращает нас к разреженным векторам, но делает их «умными».

### Альтернативы
На момент появления uniCOIL существовали:
* **BM25 (1990-е):** использует статистические веса, но не понимает синонимы.
* **DeepCT (2019):** использует BERT для предсказания весов терминов, но не умеет в Query Expansion.
* **DocT5Query (2019):** расширяет документ, генерируя возможные вопросы, но ранжирование идет через BM25.
* **COIL (2021):** для каждого токена хранился небольшой вектор, требующий кастомных индексов.
* **SPLADE (2021):** строит разреженный вектор по всему словарю BERT, но требует сложной регуляризации.

### Идея
Авторы упростили COIL, доказав, что достаточно одного скалярного веса для каждого токена, если правильно расширить документ. uniCOIL объединяет семантическое взвешивание терминов и расширение документа в единый пайплайн разреженного поиска.

### Архитектура
Модель строится на базе **Encoder-only** трансформера (обычно BERT-base).
1. **Input:** Последовательность токенов текста.
2. **Transformer Layers:** Извлечение контекстных эмбеддингов для каждого токена.
3. **Weighting Head:** Линейный слой преобразует вектор токена в скаляр.
4. **Activation:** ReLU для положительных весов.

<img src="img/img.png" width=500>

### Алгоритм обучения
Используется **Siamese Network** и Contrastive Learning:
1. Берется тройка: Query ($q$), Positive Document ($d^+$) и Hard Negative Document ($d^-$).
2. Модель прогоняет все три текста и выдает веса для каждого токена.
3. Считается Relevance Score как сумма произведений весов совпадающих токенов.
4. Минимизируется Negative Log-Likelihood (NLL).

### Алгоритм инференса
1. **Document Expansion (Offline):** К документу добавляются новые токены с помощью DocT5Query.
2. **Indexing (Offline):** Расширенный документ прогоняется через uniCOIL, токены получают вес, записываются в Inverted Index.
3. **Query Processing (Online):** Запрос прогоняется через uniCOIL для получения весов токенов.
4. **Search:** Выполняется взвешенный поиск в инвертированном индексе.

### Результаты
* **Эффективность:** uniCOIL занимает в 4-5 раз меньше места на диске, чем Dense Retrieval индексы (DPR).
* **Качество:** На MS MARCO модель показала MRR@10 около 0.351, превосходя BM25 (~0.184) и DPR (~0.31-0.34).
* **Новизна:** Переход от векторных представлений токенов к скалярным весам не снижает точность поиска, упрощая архитектуру поисковых систем.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример иллюстрации метода uniCOIL для Sparse Retrieval

import torch
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch.nn.functional as F

# Инициализация BERT модели и токенизатора
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Пример текста документа и запроса
document_text = "Deep learning models are powerful tools for data analysis."
query_text = "What are the tools for data analysis?"

# Токенизация текста
doc_tokens = tokenizer(document_text, return_tensors='pt')
query_tokens = tokenizer(query_text, return_tensors='pt')

# Получение эмбеддингов из BERT
doc_embeddings = model(**doc_tokens).last_hidden_state
query_embeddings = model(**query_tokens).last_hidden_state

# Определение линейного слоя для получения весов токенов
class WeightingHead(nn.Module):
    def __init__(self, input_dim):
        super(WeightingHead, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        # Применение линейного слоя и ReLU активации
        return F.relu(self.linear(x))

# Инициализация линейного слоя
weighting_head = WeightingHead(input_dim=doc_embeddings.size(-1))

# Получение весов токенов
doc_weights = weighting_head(doc_embeddings).squeeze(-1)
query_weights = weighting_head(query_embeddings).squeeze(-1)

# Вычисление релевантности через скалярное произведение весов совпадающих токенов
def compute_relevance_score(query_weights, doc_weights, query_tokens, doc_tokens):
    # Извлечение индексов совпадающих токенов
    query_token_ids = query_tokens['input_ids'].squeeze().tolist()
    doc_token_ids = doc_tokens['input_ids'].squeeze().tolist()
    
    # Словарь для хранения весов токенов документа
    doc_weight_dict = {token_id: weight.item() for token_id, weight in zip(doc_token_ids, doc_weights)}
    
    # Вычисление релевантности
    relevance_score = 0.0
    for token_id, weight in zip(query_token_ids, query_weights):
        if token_id in doc_weight_dict:
            relevance_score += weight.item() * doc_weight_dict[token_id]
    
    return relevance_score

# Вычисление релевантности между запросом и документом
relevance_score = compute_relevance_score(query_weights, doc_weights, query_tokens, doc_tokens)
print(f"Relevance Score: {relevance_score}")

# Примечание: В реальной системе, после получения весов, они сохраняются в инвертированный индекс
# и используются для быстрого поиска и ранжирования документов.
```

### Комментарии к коду:

1. **Токенизация и эмбеддинги:** Используется BERT для получения контекстных эмбеддингов токенов как для документа, так и для запроса.

2. **Линейный слой (Weighting Head):** Простой линейный слой с активацией ReLU используется для преобразования эмбеддингов токенов в скалярные веса. Это ключевая часть uniCOIL, которая заменяет традиционные TF-IDF веса.

3. **Вычисление релевантности:** Релевантность между запросом и документом вычисляется как сумма произведений весов совпадающих токенов. Это позволяет использовать классические инвертированные индексы для быстрого поиска.

4. **Использование ReLU:** Активация ReLU гарантирует, что веса токенов будут положительными, что необходимо для совместимости с традиционными поисковыми движками.

Этот пример иллюстрирует, как uniCOIL использует BERT для вычисления весов токенов, сохраняя при этом возможность использования классических инвертированных индексов для эффективного поиска.